# Features that do not lie

You are now the person who hands the modelling session their `X`. That matrix is the whole
contract between your section and theirs, and a feature matrix has no error messages — it either
has the right numbers in it or it quietly has the wrong ones.

A short prelude on size, then two blocks of about twelve minutes.

The same rules as the last notebook: worked example first, predict before you compute, an exercise
with the last step left for you, and a checkpoint cell so nothing downstream can strand you.
🟢 is the core, 🔵 a variation, ⬛ harder. **Finishing 🟢 is finishing.**

## Setup

> **If feedback appears blank**, trust the notebook: *Command Palette → Trust Notebook*.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")


def find_dir():
    for candidate in [Path.cwd(), Path.cwd() / "challenge_v2", *Path.cwd().parents]:
        if (candidate / "aircheck.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find aircheck.py — run this notebook from the folder it came in.")


HERE = find_dir()
sys.path.insert(0, str(HERE))
import aircheck


def find_data(filename="WDR91.parquet"):
    for root in [Path.cwd(), HERE, HERE.parent, HERE.parent.parent]:
        for path in [root / "data" / filename, root / filename]:
            if path.exists():
                return path
    raise FileNotFoundError(f"Could not find {filename} in a data/ folder.")


FINGERPRINTS = ["ECFP4", "ECFP6", "FCFP4", "FCFP6", "MACCS",
                "RDK", "AVALON", "ATOMPAIR", "TOPTOR"]

DATA = find_data()
parquet = pq.ParquetFile(DATA)


def fp_matrix(table, name):
    """One fingerprint column -> a dense (n_molecules, n_bits) array.

    Flattens the whole Arrow list column into a single buffer and reshapes, rather
    than building a Python object per molecule.
    """
    column = table.column(name).combine_chunks()
    return column.flatten().to_numpy(zero_copy_only=False).reshape(len(column), -1)


print(f"{DATA.name}  ·  {parquet.metadata.num_rows:,} rows  ·  "
      f"{DATA.stat().st_size / 1e9:.2f} GB on disk")

---
# Prelude · How big is this thing, really

**⏱ ~4 minutes. Read and run — there is one prediction and no exercise.**

Parquet keeps a metadata footer describing every column, so you can size a dataset up before
reading a single value.

In [ ]:
peek = pa.Table.from_batches([next(parquet.iter_batches(batch_size=2, columns=FINGERPRINTS))])
widths = {fp: len(peek.column(fp)[0]) for fp in FINGERPRINTS}

print(pd.Series(widths).to_string())
print(f"\ntotal bits across the nine: {sum(widths.values()):,}")

## 🔮 Predict

**If you loaded all nine fingerprint columns for all 375,595 compounds as a dense float32 array,
how many gigabytes of RAM would that need?**

The file is 0.66 GB on disk. Commit a number.

In [ ]:
# aircheck.predict("b0_gb", ...)
aircheck.predict("b0_gb", 2)  # placeholder guess, not the answer -- the aircheck.check cell below is what is graded

In [ ]:
dense_gb = parquet.metadata.num_rows * sum(widths.values()) * 4 / 1e9
print(f"{dense_gb:.1f} GB")
print()

aircheck.check("b0_gb", dense_gb)

So no loader in this session ever reads the whole file. Two patterns cover everything:

- **Narrow columns for every row.** The label, the library and the building-block IDs are a few
  megabytes across all 375,595 compounds. Read them once, whole, and keep them.
- **Fingerprints in batches.** Those are the 25 GB. Walk them one batch at a time, keep the numbers
  you need, and let each batch go.

In [ ]:
# Pattern 1: every row, but only the narrow columns. No fingerprints, so this is a ~1 s read.
META_COLS = ["LABEL", "LIBRARY_ID", "BB1_ID", "BB2_ID", "BB3_ID", "MW", "ALOGP"]
table = pq.read_table(DATA, columns=META_COLS)
y = table.column("LABEL").to_numpy()

libs = pd.Series(table.column("LIBRARY_ID").to_pandas()).str.split("-").str[0]
print(f"{table.num_rows:,} molecules · hit rate {y.mean():.2%} · {libs.nunique()} libraries · "
      f"{table.nbytes / 1e6:.0f} MB in memory")

### 🔵 One dtype trap worth two minutes

`uint8` is an easy 4x saving over float32 for small counts. Somebody tested it on ECFP4 and it was
fine. Run this and watch what happens to AVALON.

In [ ]:
# Pattern 2: the two fingerprint columns, every row, one batch at a time. Each batch is
# converted, summarised, and dropped; only four numbers per fingerprint survive the loop.
largest, largest_as_uint8, altered = {}, {}, {}
for batch in parquet.iter_batches(batch_size=50_000, columns=["ECFP4", "AVALON"]):
    tbl = pa.Table.from_batches([batch])
    for fp in ["ECFP4", "AVALON"]:
        counts = fp_matrix(tbl, fp)
        largest[fp] = max(largest.get(fp, 0), int(counts.max()))
        largest_as_uint8[fp] = max(largest_as_uint8.get(fp, 0), int(counts.astype(np.uint8).max()))
        altered[fp] = altered.get(fp, 0) + int((counts > 255).sum())

for fp in ["ECFP4", "AVALON"]:
    print(f"{fp:<6} largest count {largest[fp]:>4}   as uint8: {largest_as_uint8[fp]:>4}")
print()
print(f"{largest['AVALON']} on its own becomes {np.array([largest['AVALON']]).astype(np.uint8)[0]} "
      "— it wrapped around.")
print(f"{altered['AVALON']:,} values were silently altered, with no warning at all.")
print("\nNote the second trap: the max of the converted array is not the conversion of the max,")
print("so even the summary statistic you would sanity-check against has been scrambled.")

---
# Block 3 · Building blocks that are real

**⏱ ~12 minutes**

DEL compounds are built combinatorially: pick a BB1, a BB2, a BB3, react them. So the question a
chemist can actually act on is not "which compound scored well" but **which building blocks does
this protein like**.

We will work inside library **L04** — 40,148 compounds, 11.5% hit rate.

## The cell below is wrong. The bug is in the ranking, not the code.

It runs. It produces a plausible-looking table. Look at the `n` column.

In [ ]:
LIB = "L04"
meta = table.select(["LIBRARY_ID", "BB1_ID", "BB2_ID", "BB3_ID", "LABEL"]).to_pandas()
meta["LIB"] = meta["LIBRARY_ID"].str.split("-").str[0]
sub = meta[meta.LIB == LIB]

blocks = sub.groupby("BB3_ID").agg(n=("LABEL", "size"), hits=("LABEL", "sum"))
blocks["hit_rate"] = blocks.hits / blocks.n

print(f"library {LIB}: {len(sub):,} compounds, hit rate {sub.LABEL.mean():.1%}, "
      f"{len(blocks):,} distinct BB3 blocks\n")
print("Top 5 building blocks by hit rate:")
display(blocks.sort_values(["hit_rate", "n"], ascending=[False, False]).head(5))

**The debugging routine for this session** — reuse it every time something looks odd:

1. check the **dtype** — is it what you think it is?
2. check for **nulls** — is the column even populated?
3. check **one row by hand** — does the number survive contact with a human?
4. check the **denominator** — how many observations is this built on?

Step 4 is the one that fires here.

In [ ]:
top = blocks.sort_values(["hit_rate", "n"], ascending=[False, False]).iloc[0]
print(top.to_string(), "\n")

# naive_n = ...      # <- how many compounds is the top-ranked block actually built on?
naive_n = int(top.n)
aircheck.check("b3_naive_n", naive_n)

## 🔮 Predict

Everything so far has looked at **one position at a time**. But a compound is three blocks reacted
together, and a binding site does not necessarily care about them independently.

**Take the best BB1–BB3 *pair* in this library (seen at least 20 times). What hit rate does it
reach?** The best single block managed 45%.

In [ ]:
# aircheck.predict("b3_pair_rate", ...)      # a fraction between 0 and 1
aircheck.predict("b3_pair_rate", 0.3)      # a fraction between 0 and 1

## 🟢 Exercise — disynthon aggregation

This is the standard preprocessing step in DEL machine learning. McCloskey et al., in the reference
paper for the field, describe it exactly this way:

> *"for a three cycle library of the form A-B-C, sums aggregating counts for the A-B, A-C, and B-C
> disynthons were generated"*

Group by **two** building-block columns at once instead of one.

In [ ]:
# pairs = ...      # <- group by BB1_ID *and* BB3_ID together; count n and hits as before
pairs = sub.groupby(["BB1_ID", "BB3_ID"]).agg(n=("LABEL", "size"), hits=("LABEL", "sum"))


pairs["hit_rate"] = pairs.hits / pairs.n
supported = pairs[pairs.n >= 20].sort_values(["hit_rate", "n"], ascending=[False, False])

display(supported.head(5).round(3))
print()

aircheck.check("b3_pair_rate", supported.iloc[0].hit_rate)

## 🟢 Exercise — look inside the winner

That pair fixes BB1 and BB3 and says nothing about BB2. So what is in the middle position of those
compounds?

In [ ]:
b1_id, b3_id = supported.index[0]
group = sub[(sub.BB1_ID == b1_id) & (sub.BB3_ID == b3_id)]

# n_distinct_bb2 = ...     # <- how many different BB2 blocks appear in this group?
n_distinct_bb2 = int(group.BB2_ID.nunique())
print(f"BB1 {b1_id} + BB3 {b3_id}: {len(group)} compounds, {int(group.LABEL.sum())} hits")
print(f"distinct BB2 blocks among them: {n_distinct_bb2}")
print()
print(f"  BB1 {b1_id} on its own: {sub[sub.BB1_ID == b1_id].LABEL.mean():.0%}")
print(f"  BB3 {b3_id} on its own: {sub[sub.BB3_ID == b3_id].LABEL.mean():.0%}")
print()

aircheck.check("b3_bb2", n_distinct_bb2)

### 💭 Concept check

You aggregated single blocks, then pairs. The obvious next step is to aggregate all three.

**Why does DEL preprocessing stop at pairs?**

- **a)** three-way groups are too rare to reach a useful count
- **b)** a three-block group contains exactly one compound
- **c)** the number of groups makes it too slow to compute
- **d)** BB2–BB3 pairs are not chemically meaningful

In [ ]:
# aircheck.mcq("b3_stop", ...)
aircheck.mcq("b3_stop", "b")

## Before Block 4 — could the fingerprint have found that pair?

You found the pair with a `groupby`. A model does not get a `groupby` — it gets a fingerprint. So
the question that decides what Block 4 has to build is: **does ECFP4 encode the building blocks?**

Measured on L04 (the ⬛ section at the end of this notebook draws the picture, ~30 s):

| compounds carrying… | land in how many of 56 chemical-space clusters | hit rate |
|---|---|---|
| a random set of 1,668 | 46 | 9.3% |
| BB1 `0025` | **5** | 9.6% |
| BB3 `2549` | **27** | 32.0% |

ECFP4 is not blind to either block, but it sees BB1 about five times more sharply than BB3 — and
BB1 is the position that carries no signal (below the 11.5% library baseline), while BB3 is the one
that does. **The position the fingerprint resolves most clearly is the one that matters least.**

That is why Block 4 builds explicit building-block features: the representation you already have
puts its resolution in the wrong place, and a model given only fingerprints is not being asked a
fair question.

### ⬛ If you got here early

Run the cell below to try the same aggregation in three other libraries, then change the position
pair (`("BB1_ID", "BB2_ID")`) and see whether the pattern moves.

Exactly-100% pairs are rare — two in the whole file. **Near-saturation is not.** That is the more
useful finding: strong two-block combinations are a general property of this data, not a one-off,
which is precisely why pair-level aggregation is the field's standard preprocessing step rather
than a trick.

In [ ]:
def best_pair(lib, positions=("BB1_ID", "BB3_ID"), min_n=20):
    """Best disynthon in one library. Change the library and re-run."""
    part = meta[meta.LIB == lib]
    g = part.groupby(list(positions)).agg(n=("LABEL", "size"), hits=("LABEL", "sum"))
    g = g[g.n >= min_n]
    if g.empty:
        print(f"{lib}: no {'-'.join(positions)} pair reaches {min_n} compounds")
        return
    g["hit_rate"] = g.hits / g.n
    winner = g.sort_values(["hit_rate", "n"], ascending=[False, False]).iloc[0]
    print(f"{lib}: best pair {g.sort_values(['hit_rate','n'], ascending=[False,False]).index[0]} "
          f"-> {winner.hit_rate:.0%} of {int(winner.n)} compounds "
          f"(library baseline {part.LABEL.mean():.1%})")


for lib in ["L04", "L23", "L30", "L49"]:
    best_pair(lib)

---
# Block 4 · The feature that carries the label

**⏱ ~12 minutes**

The building-block columns are the most valuable thing in this file — you would never have this
information for an ordinary compound collection. The obvious way to use them is **target
encoding**: replace each block with its hit rate.

It is a genuinely good feature, and it is also the easiest way to hand the modelling session a
matrix that scores brilliantly and means nothing.

## Worked example — namespace the IDs first

Building block IDs restart from `0001` in every library, so `0043` in L23 and `0043` in L49 are
unrelated chemicals. Grouping on the raw ID averages them together.

In [ ]:
frame = table.select(["LIBRARY_ID", "BB1_ID", "BB2_ID", "BB3_ID", "LABEL"]).to_pandas()
frame["LIB"] = frame["LIBRARY_ID"].str.split("-").str[0]

# frame["BB1_ns"] = ...      # <- prefix each BB1 ID with its library
frame["BB1_ns"] = frame["LIB"] + "_" + frame["BB1_ID"]

shared = frame.groupby("BB1_ID")["LIB"].nunique()
print(f"raw BB1 vocabulary        {frame.BB1_ID.nunique():>6,}")
print(f"library-namespaced        {frame.BB1_ns.nunique():>6,}")
print(f"raw IDs used by more than one library: {int((shared > 1).sum()):,}")
print()

aircheck.check("b4_ns_vocab", frame.BB1_ns.nunique())

## Now encode it the obvious way

In [ ]:
frame["BB1_ns"] = frame["LIB"] + "_" + frame["BB1_ID"]      # in case you skipped above

group_stats = frame.groupby("BB1_ns")["LABEL"].agg(["sum", "count"])
raw_rate = frame["BB1_ns"].map(group_stats["sum"] / group_stats["count"]).to_numpy()

gap_raw = raw_rate[y == 1].mean() - raw_rate[y == 0].mean()
print(f"mean feature value  —  hits {raw_rate[y == 1].mean():.3f}   "
      f"non-hits {raw_rate[y == 0].mean():.3f}   gap {gap_raw:.3f}")
print("\nBeautiful separation. Suspiciously beautiful separation.")

## 🔮 Predict

**What fraction of rows ended up with a feature value exactly equal to their own label?**

0 would mean none of them; 1 would mean every single row.

In [ ]:
# aircheck.predict("b4_selfleak", ...) # placeholder guess, not the answer -- the aircheck.check cell below is what is graded
aircheck.predict("b4_selfleak", 0.03)

## 🟢 Exercise — measure it

In [ ]:
group_n = frame["BB1_ns"].map(group_stats["count"]).to_numpy()

# self_leak = ...      # <- fraction of rows whose feature value IS their own label
self_leak = float((raw_rate == y).mean())
print(f"synthons appearing exactly once: {int((group_stats['count'] == 1).sum()):,} "
      f"of {len(group_stats):,}")
print(f"rows sitting in a group of one:  {int((group_n == 1).sum()):,}")
print()

aircheck.check("b4_selfleak", self_leak)

## 🟢 Exercise — fix it

Two changes, and they fix different problems:

- **Smoothing** pulls small groups toward the global rate, so a block seen twice cannot claim a hit
  rate of 1.0. It fixes the **variance**.
- **Leave-one-out** removes the row's own label from its own group's statistic, so a compound never
  votes for itself. It fixes the **leak**.

You need both. Smoothing alone still lets every row see its own answer.

In [ ]:
SMOOTHING = 10
global_rate = y.mean()
group_sum = frame["BB1_ns"].map(group_stats["sum"]).to_numpy()

# smoothed = ...    # <- smoothing only: pull small groups toward global_rate
# loo = ...         # <- smoothing AND leave-one-out: take this row out of its own group

smoothed = (group_sum + SMOOTHING * global_rate) / (group_n + SMOOTHING)
loo = (group_sum - y + SMOOTHING * global_rate) / (group_n - 1 + SMOOTHING)

gap_loo = loo[y == 1].mean() - loo[y == 0].mean()

comparison = pd.DataFrame({
    "encoding": ["raw", "smoothed", "smoothed + leave-one-out"],
    "non-hits": [raw_rate[y == 0].mean(), smoothed[y == 0].mean(), loo[y == 0].mean()],
    "hits": [raw_rate[y == 1].mean(), smoothed[y == 1].mean(), loo[y == 1].mean()],
    "gap": [gap_raw, smoothed[y == 1].mean() - smoothed[y == 0].mean(), gap_loo],
    "= own label": [(raw_rate == y).mean(), (smoothed == y).mean(), (loo == y).mean()],
}).set_index("encoding")
display(comparison.round(4))
print()

aircheck.check("b4_gap_loo", gap_loo)

In [ ]:
# 🧭 Checkpoint
smoothed = (group_sum + SMOOTHING * global_rate) / (group_n + SMOOTHING)
loo = (group_sum - y + SMOOTHING * global_rate) / (group_n - 1 + SMOOTHING)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, (name, values) in zip(axes, [("raw", raw_rate), ("smoothed", smoothed),
                                     ("smoothed + leave-one-out", loo)]):
    ax.hist(values[y == 0], bins=40, alpha=0.7, label="non-hit", color="#b9c0cc", density=True)
    ax.hist(values[y == 1], bins=40, alpha=0.7, label="hit", color="#d62728", density=True)
    ax.set(title=name, xlabel="BB1 synthon feature", yscale="log")
axes[0].set_ylabel("density (log)")
axes[0].legend()
plt.tight_layout()
plt.show()

aircheck.checkpoint("Left panel: two spikes, at 0 and 1, each almost entirely one class. "
               "That is not a feature separating the classes — that is the label wearing a hat.")

### 💭 Concept check

You now have an honest feature. **What has to happen when you hand it to the modelling session?**

- **a)** nothing — leave-one-out fixed it, the matrix is safe to hand over as-is
- **b)** they must recompute the synthon rates inside each training fold
- **c)** drop the synthon features; they are too dangerous to use
- **d)** raise the smoothing until the leak disappears

In [ ]:
# aircheck.mcq("b4_handoff", ...)
aircheck.mcq("b4_handoff", "a")

### ⬛ If you got here early — assemble the matrix

Everything above, as one function. Nothing here is checked; run it and read what comes out.

In [ ]:
def synthon_rates(frame, position, labels, smoothing=10):
    """Leave-one-out, smoothed hit rate for one building-block position."""
    key = f"{position}_ns"
    stats_ = frame.groupby(key)["LABEL"].agg(["sum", "count"])
    total = frame[key].map(stats_["sum"]).to_numpy()
    n = frame[key].map(stats_["count"]).to_numpy()
    return (total - labels + smoothing * labels.mean()) / (n - 1 + smoothing)


def featurize(path, n_rows=20000, fingerprint="ECFP4", min_bit_frequency=0.001):
    """Fingerprint bits + library one-hot + honest synthon rates + scaled properties."""
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    pf = pq.ParquetFile(path)
    cols = [fingerprint, "LABEL", "LIBRARY_ID", "BB1_ID", "BB2_ID", "BB3_ID", "MW", "ALOGP"]
    tbl = pa.Table.from_batches([next(pf.iter_batches(batch_size=n_rows, columns=cols))])

    labels = tbl.column("LABEL").to_numpy()
    meta_ = tbl.select(["LIBRARY_ID", "BB1_ID", "BB2_ID", "BB3_ID", "MW", "ALOGP"]).to_pandas()
    meta_["LIB"] = meta_["LIBRARY_ID"].str.split("-").str[0]
    meta_["LABEL"] = labels
    for position in ["BB1", "BB2", "BB3"]:
        meta_[f"{position}_ns"] = meta_["LIB"] + "_" + meta_[f"{position}_ID"]

    bits = (fp_matrix(tbl, fingerprint) > 0).astype(np.float32)
    bits = bits[:, bits.mean(0) >= min_bit_frequency]

    rates = np.column_stack([synthon_rates(meta_, p, labels) for p in ["BB1", "BB2", "BB3"]])
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=True).fit_transform(meta_[["LIB"]])
    props = StandardScaler().fit_transform(meta_[["MW", "ALOGP"]].astype(np.float32))

    blocks_ = [sp.csr_matrix(bits), onehot,
               sp.csr_matrix(rates.astype(np.float32)), sp.csr_matrix(props.astype(np.float32))]
    names = [f"{fingerprint} bits ({bits.shape[1]})", f"library one-hot ({onehot.shape[1]})",
             "synthon rates, leave-one-out (3)", "MW / ALOGP (2)"]
    return sp.hstack(blocks_, format="csr"), labels, names


X, y_out, names = featurize(DATA)
sparse_mb = (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6
print("blocks:", *names, sep="\n  ")
print(f"\nX: {X.shape[0]:,} x {X.shape[1]:,}  ({sparse_mb:.1f} MB sparse, "
      f"vs {X.shape[0] * X.shape[1] * 4 / 1e6:.0f} MB dense)")

### ⬛ If you got here early — draw what the fingerprint actually sees

This is the picture behind the table in the bridge before Block 4. It is a UMAP of L04's ECFP4 fingerprints — the same diagnostic as notebook 1, pointed at one
library instead of all 39. It is computed here, with the same settings as before
(`n_neighbors=20, min_dist=0.1, metric="jaccard", random_state=42`); the mechanics are explained
in notebook 1 under *What UMAP does*.

Two things are different this time:

- **The fingerprints are read in batches.** L04's 40,148 compounds are scattered through the file,
  so the cell walks the `ECFP4` column 100,000 rows at a time and keeps only the L04 rows. That is
  a pass over the whole file, but only one column of it, and never more than one batch in memory.
- **The sample is deliberate, not random.** Every compound carrying the winning BB1, the winning
  BB3, or both goes in, plus 3,500 random L04 compounds as background. The question is where the
  building blocks land, so the building blocks have to be there.

The embedding falls into 56 clusters. A randomly chosen set of ~1,700 compounds is spread over
about 46 of them; the compounds carrying BB1 `0025` sit in just 5, one contiguous region; the
compounds carrying BB3 `2549` sit in 27. Run the cell and see it.

In [ ]:
import time
import warnings

# Same three harmless warnings as notebook 1: tqdm wanting ipywidgets, no inverse_transform for
# Jaccard, and a fixed seed disabling parallelism.
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message="gradient function is not yet implemented")
warnings.filterwarnings("ignore", message="n_jobs value .* overridden")

import umap
import hdbscan

N_BACKGROUND = 3500
UMAP_KWARGS = dict(n_components=2, n_neighbors=20, min_dist=0.1,
                   metric="jaccard", random_state=42)

# L04's ECFP4, one batch at a time: LIBRARY_ID decides which rows to keep, and only those rows'
# fingerprints are ever materialised.
t0 = time.perf_counter()
chunks = []
for batch in parquet.iter_batches(batch_size=100_000, columns=["LIBRARY_ID", "ECFP4"]):
    tbl = pa.Table.from_batches([batch])
    mask = (tbl.column("LIBRARY_ID").to_pandas().str.split("-").str[0] == LIB).to_numpy()
    if mask.any():
        chunks.append((fp_matrix(tbl, "ECFP4")[mask] > 0).astype(np.uint8))
l04_binary = np.vstack(chunks)
l04 = sub.reset_index(drop=True)          # Block 3's L04 rows, in file order, same as the batches
assert len(l04) == len(l04_binary)
print(f"read {LIB}'s ECFP4: {l04_binary.shape[0]:,} x {l04_binary.shape[1]:,} bits, "
      f"{time.perf_counter() - t0:.0f} s")

# Four groups: the winning pair, each of its two blocks on its own, and everything else.
l04["GROUP"] = np.select(
    [(l04.BB1_ID == b1_id) & (l04.BB3_ID == b3_id), l04.BB3_ID == b3_id, l04.BB1_ID == b1_id],
    ["pair", f"BB3 {b3_id} only", f"BB1 {b1_id} only"], default="other")

# Every compound in the three named groups, plus a random background.
rng = np.random.default_rng(0)
other = np.where(l04.GROUP == "other")[0]
keep = np.concatenate([np.where(l04.GROUP != "other")[0],
                       rng.choice(other, min(N_BACKGROUND, len(other)), replace=False)])
keep.sort()

t0 = time.perf_counter()
embedding = umap.UMAP(**UMAP_KWARGS).fit_transform(l04_binary[keep].astype(np.float32))
print(f"UMAP on {len(keep):,} compounds: {time.perf_counter() - t0:.0f} s")

space = l04.iloc[keep].reset_index(drop=True)
space["UMAP1"], space["UMAP2"] = embedding[:, 0], embedding[:, 1]
space["CLUSTER"] = hdbscan.HDBSCAN(min_cluster_size=25).fit_predict(embedding)
bb1, bb3 = b1_id, b3_id

In [ ]:
clustered = space[space.CLUSTER >= 0]          # HDBSCAN leaves some points unassigned

rows = space.groupby("GROUP").agg(n=("LABEL", "size"), hit_rate=("LABEL", "mean"))
rows["clusters"] = clustered.groupby("GROUP")["CLUSTER"].nunique()

# Always ask what the number would be if the group meant nothing: a set this size, drawn at random,
# spans how many clusters? Without this row, "27" is impossible to interpret.
pool = clustered[clustered.GROUP == "other"]
size = int(rows.loc[f"BB1 {bb1} only", "n"])
rows.loc[f"random {size:,}"] = [size, pool.LABEL.mean(),
                                np.median([pool.sample(size, random_state=r).CLUSTER.nunique()
                                           for r in range(10)])]
rows[["n", "clusters"]] = rows[["n", "clusters"]].astype(int)

print(f"{clustered.CLUSTER.nunique()} clusters in the embedding\n")
display(rows.loc[[f"BB1 {bb1} only", f"BB3 {bb3} only", "pair", f"random {size:,}"]].round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

axes[0].scatter(space.UMAP1, space.UMAP2, s=6, c="#b9c0cc", alpha=0.5)
hits = space[space.LABEL == 1]
axes[0].scatter(hits.UMAP1, hits.UMAP2, s=10, c="#d62728", alpha=0.8)
axes[0].set_title(f"coloured by activity — {len(hits):,} hits")

background = space[space.GROUP == "other"]
axes[1].scatter(background.UMAP1, background.UMAP2, s=5, c="#d9dce2", alpha=0.6)
for group, colour in [(f"BB1 {bb1} only", "#4c72b0"),
                      (f"BB3 {bb3} only", "#dd8452"),
                      ("pair", "#2e7d4f")]:
    part = space[space.GROUP == group]
    axes[1].scatter(part.UMAP1, part.UMAP2, s=10, c=colour, alpha=0.85, label=group)
axes[1].legend(markerscale=2, fontsize=9, loc="lower left")
axes[1].set_title("coloured by building block")

for ax in axes:
    ax.set(xlabel="UMAP1", ylabel="UMAP2")
plt.tight_layout()
plt.show()

aircheck.checkpoint("Blue is one streak. Orange is scattered over a dozen islands. ECFP4 encodes "
               "both blocks — but it encodes BB1 about five times more sharply, and BB1 is "
               "not the position carrying the activity.")

Read the `clusters` column against its last row. Both blocks are more concentrated than chance:
random is ~46, BB3 `2549` is 27, BB1 `0025` is 5. So ECFP4 is **not blind to BB3** — it just
encodes BB1 far more sharply. ECFP4 hashes circular atom environments, and in this library those
environments are dominated by the BB1 position.

Now put that next to the `hit_rate` column. **The position the fingerprint resolves most sharply is
the one that matters least.** BB1 `0025` runs at 9.6%, *below* the library's 11.5% baseline. BB3
`2549` — the block smeared across 27 clusters — runs at 32.0%.

That is the argument for Block 4: **you need explicit building-block features precisely because the
representation you already have puts its resolution in the wrong place.** A model given only
fingerprints is not being asked a fair question.

Two more things in the picture worth saying out loud:

- **The hits do not live in one place.** They are spread across most of the library's clusters, so
  "find the active region of chemical space" is not the job you are handing the model.
- **The pair sits inside BB1 `0025`'s territory, which is below baseline.** A 100% pocket inside a
  cold region — the same shape as the molecular-weight reversal in notebook 1.

### What a UMAP is for, and what it is not

**It is a diagnostic, not a result.** You use it to find structure, then you leave the picture and
put a number on the structure. That is exactly what just happened: the plot suggested "one position
dominates the geometry", and the `clusters` column measured it as 5 against 27 against a null of 46.
Notebook 1 did the same thing, going from library-coloured blobs to 82.6% purity. Never report the
picture on its own, and never report a count like "27" without the null next to it.

Three things that are true of every UMAP and get forgotten anyway:

- **Distance between clusters means nothing.** Only local neighbourhoods are roughly preserved. Two
  blobs far apart here are not more different than two blobs side by side.
- **Cluster size means nothing either** — it is an artefact of the layout, not a count.
- **It is stochastic and hyperparameter-dependent.** `n_neighbors=20, min_dist=0.1,
  random_state=42` produces this picture; other settings produce a different one from the same data.

One more that is specific to fingerprints: the metric is **Jaccard, not Euclidean**. On sparse binary
vectors Euclidean distance largely measures how many bits are set — that is, molecule size. Jaccard
(Tanimoto) asks the question a chemist means.

---
## Where you got to

In [ ]:
aircheck.progress()

### The handover note

Three things about that matrix are invisible from the matrix itself, and the next session needs all
three in writing:

1. **The synthon columns were computed from the labels.** Leave-one-out makes them honest on *these*
   rows and nowhere else. Whoever splits the data has to recompute them inside their own training
   fold — otherwise the leak comes straight back, and worse, because now it looks fixed.
2. **The bit filter and the scaler were fitted on these 20,000 rows too.** Same rule.
3. **The library one-hot is a confounder handed over deliberately.** It is genuinely predictive and
   genuinely not chemistry. Notebook 1 measured a 16x hit-rate spread and 83% library-pure clusters;
   they should group their folds by library, and check what the model does without that column.

The general form: **any feature that looked at the label, or at the rest of the dataset, is a
modelling decision wearing a feature-engineering costume.** You do not get to make it quietly.